In [1]:
!pip install -q sentence-transformers

In [2]:
from sentence_transformers import SentenceTransformer

model_name = "all-MiniLM-L6-v2"

model = SentenceTransformer(model_name)

print("="*50)
print("Embedding Model Details")
print("="*50)

print("Model Name :", model_name)
print("Embedding Dimension :", model.get_sentence_embedding_dimension())

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Embedding Model Details
Model Name : all-MiniLM-L6-v2
Embedding Dimension : 384


/tmp/ipykernel_1741/3950025401.py:12: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  print("Embedding Dimension :", model.get_sentence_embedding_dimension())


In [3]:
chunks = [
    "Employees receive 12 casual leaves annually.",
    "Employees receive 15 sick leaves annually.",
    "Employees may work from home twice per week.",
    "Travel expenses are reimbursed within 30 days.",
    "All employees are covered under company medical insurance."
]

chunk_embeddings = model.encode(chunks)

In [4]:
print("="*50)
print("Chunk Embeddings")
print("="*50)

for i, chunk in enumerate(chunks):
    print(f"\nChunk {i+1}")
    print(chunk)
    print("Embedding Shape :", chunk_embeddings[i].shape)

Chunk Embeddings

Chunk 1
Employees receive 12 casual leaves annually.
Embedding Shape : (384,)

Chunk 2
Employees receive 15 sick leaves annually.
Embedding Shape : (384,)

Chunk 3
Employees may work from home twice per week.
Embedding Shape : (384,)

Chunk 4
Travel expenses are reimbursed within 30 days.
Embedding Shape : (384,)

Chunk 5
All employees are covered under company medical insurance.
Embedding Shape : (384,)


In [5]:
print("="*50)
print("First 20 Embedding Values")
print("="*50)

print(chunk_embeddings[0][:20])

First 20 Embedding Values
[ 0.0618362   0.01376683  0.03366624  0.0186107   0.03135883  0.06788085
 -0.01135737 -0.01733116 -0.07070484  0.01901567  0.1098766   0.05092814
 -0.0489678  -0.04620624 -0.03665631  0.00247606 -0.06287521  0.00541349
  0.03131726 -0.07714854]


In [6]:
queries = [
    "How many casual leaves are allowed?",
    "Can employees work remotely?",
    "What is the travel reimbursement process?",
    "Do employees have medical insurance?"
]

query_embeddings = model.encode(queries)

In [7]:
print("="*50)
print("Query Embeddings")
print("="*50)

for i, query in enumerate(queries):
    print(f"\nQuery {i+1}")
    print(query)
    print("Embedding Shape :", query_embeddings[i].shape)

Query Embeddings

Query 1
How many casual leaves are allowed?
Embedding Shape : (384,)

Query 2
Can employees work remotely?
Embedding Shape : (384,)

Query 3
What is the travel reimbursement process?
Embedding Shape : (384,)

Query 4
Do employees have medical insurance?
Embedding Shape : (384,)


In [8]:
from sklearn.metrics.pairwise import cosine_similarity
import pandas as pd

results = []

for q_idx, query_embedding in enumerate(query_embeddings):

    similarities = cosine_similarity(
        [query_embedding],
        chunk_embeddings
    )[0]

    for c_idx, score in enumerate(similarities):

        results.append([
            queries[q_idx],
            chunks[c_idx],
            round(score,4)
        ])

df = pd.DataFrame(
    results,
    columns=["Query","Chunk","Similarity"]
)

df

,Query,Chunk,Similarity
0,How many casual leaves are allowed?,Employees receive 12 casual leaves annually.,0.6552
1,How many casual leaves are allowed?,Employees receive 15 sick leaves annually.,0.4110
2,How many casual leaves are allowed?,Employees may work from home twice per week.,0.1763
3,How many casual leaves are allowed?,Travel expenses are reimbursed within 30 days.,0.0493
4,How many casual leaves are allowed?,All employees are covered under company medica...,0.0472
5,Can employees work remotely?,Employees receive 12 casual leaves annually.,0.1963
6,Can employees work remotely?,Employees receive 15 sick leaves annually.,0.2300
7,Can employees work remotely?,Employees may work from home twice per week.,0.4834
8,Can employees work remotely?,Travel expenses are reimbursed within 30 days.,-0.0254
9,Can employees work remotely?,All employees are covered under company medica...,0.3006


In [9]:
print("="*50)
print("Most Similar Chunk")
print("="*50)

for q_idx, query_embedding in enumerate(query_embeddings):

    similarities = cosine_similarity(
        [query_embedding],
        chunk_embeddings
    )[0]

    best_index = similarities.argmax()

    print("\nQuery:")
    print(queries[q_idx])

    print("\nMost Similar Chunk:")
    print(chunks[best_index])

    print(
        "\nSimilarity Score:",
        round(similarities[best_index],4)
    )

    print("-"*50)

Most Similar Chunk

Query:
How many casual leaves are allowed?

Most Similar Chunk:
Employees receive 12 casual leaves annually.

Similarity Score: 0.6552
--------------------------------------------------

Query:
Can employees work remotely?

Most Similar Chunk:
Employees may work from home twice per week.

Similarity Score: 0.4834
--------------------------------------------------

Query:
What is the travel reimbursement process?

Most Similar Chunk:
Travel expenses are reimbursed within 30 days.

Similarity Score: 0.7255
--------------------------------------------------

Query:
Do employees have medical insurance?

Most Similar Chunk:
All employees are covered under company medical insurance.

Similarity Score: 0.803
--------------------------------------------------


In [10]:
pair1 = [
    "Employees receive 12 casual leaves.",
    "Workers are entitled to 12 annual leaves."
]

pair2 = [
    "Employees receive 12 casual leaves.",
    "Travel expenses are reimbursed within 30 days."
]

emb1 = model.encode(pair1)
emb2 = model.encode(pair2)

score1 = cosine_similarity(
    [emb1[0]],
    [emb1[1]]
)[0][0]

score2 = cosine_similarity(
    [emb2[0]],
    [emb2[1]]
)[0][0]

print("Pair 1 Similarity :", round(score1,4))
print("Pair 2 Similarity :", round(score2,4))

Pair 1 Similarity : 0.771
Pair 2 Similarity : 0.175


In [11]:
from sklearn.decomposition import PCA
import matplotlib.pyplot as plt

all_texts = chunks + queries

all_embeddings = model.encode(all_texts)

pca = PCA(n_components=2)

reduced = pca.fit_transform(all_embeddings)